In [0]:
import pyspark.sql.functions as F

In [0]:
spark.sql("USE CATALOG mvp_saude")

DataFrame[]

In [0]:
df_fato = spark.table("mvp_saude.silver.sih_cleaned")
df_muni = spark.table("mvp_saude.silver.dim_municipios")
df_cid = spark.table("mvp_saude.silver.dim_cid10")

In [0]:
df_fato_treated = df_fato \
    .withColumn("cod_muni_6", F.substring(F.trim(F.col("cod_municipio").cast("string")), 1, 6)) \
    .withColumn("cod_cid_clean", F.upper(F.trim(F.regexp_replace(F.col("cod_cid_principal").cast("string"), "\\.", "")))) \
    .withColumn("cod_cid_3dig", F.substring(F.col("cod_cid_clean"), 1, 3))

df_silver_enriquecida = df_fato_treated.alias("f") \
    .join(df_muni.alias("m"), F.col("f.cod_muni_6") == F.col("m.cod_municipio"), "left") \
    .join(df_cid.alias("c1"), F.col("f.cod_cid_clean") == F.col("c1.cod_cid_principal"), "left") \
    .join(df_cid.alias("c2"), F.col("f.cod_cid_3dig") == F.col("c2.cod_cid_principal"), "left") \
    .select(
        F.coalesce(F.col("m.uf_sigla"), F.lit("RS")).alias("uf_sigla"),
        F.col("f.cod_muni_6").alias("cod_municipio"),
        F.coalesce(
            F.col("m.nome_municipio"), 
            F.concat(F.lit("Município IBGE "), F.col("f.cod_muni_6"))
        ).alias("nome_municipio"),
        F.col("f.cod_cid_clean").alias("cod_cid_principal"),
        F.coalesce(
            F.col("c1.descricao_cid"), 
            F.col("c2.descricao_cid"), 
            F.concat(F.lit("Diagnóstico CID-10 ("), F.col("f.cod_cid_clean"), F.lit(")"))
        ).alias("descricao_cid"),
        F.col("f.val_total_internacao"),
        F.col("f.dias_permanencia"),
        F.col("f.ind_obito")
    )

# Resumo Geral

In [0]:
df_gold_geral = df_silver_enriquecida.groupBy("uf_sigla").agg(
    F.count("*").alias("total_internacoes"),
    F.round(F.sum("val_total_internacao"), 2).alias("custo_total_rs"),
    F.round(F.avg("val_total_internacao"), 2).alias("custo_medio_internacao"),
    F.round(F.avg("dias_permanencia"), 2).alias("media_dias_permanencia"),
    F.sum("ind_obito").alias("total_obitos"),
    F.round((F.sum("ind_obito") / F.count("*")) * 100, 2).alias("taxa_mortalidade_pct")
).withColumn("dh_carga_gold", F.current_timestamp())

df_gold_geral.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.gold.gold_resumo_geral_rs")

# Internações por Município

In [0]:
df_gold_municipio = df_silver_enriquecida.groupBy("uf_sigla", "cod_municipio", "nome_municipio").agg(
    F.count("*").alias("total_internacoes"),
    F.round(F.sum("val_total_internacao"), 2).alias("custo_total_municipio"),
    F.round(F.avg("val_total_internacao"), 2).alias("custo_medio_internacao"),
    F.round(F.avg("dias_permanencia"), 2).alias("media_dias_permanencia"),
    F.sum("ind_obito").alias("total_obitos"),
    F.round((F.sum("ind_obito") / F.count("*")) * 100, 2).alias("taxa_mortalidade_pct")
).orderBy(F.col("total_internacoes").desc()) \
 .withColumn("dh_carga_gold", F.current_timestamp())

df_gold_municipio.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.gold.gold_internacoes_por_municipio")

# Internações por diagnóstico

In [0]:
df_gold_cid = df_silver_enriquecida.groupBy("cod_cid_principal", "descricao_cid").agg(
    F.count("*").alias("total_internacoes"),
    F.round(F.sum("val_total_internacao"), 2).alias("custo_total_cid"),
    F.round(F.avg("val_total_internacao"), 2).alias("custo_medio_internacao"),
    F.round(F.avg("dias_permanencia"), 2).alias("media_dias_permanencia"),
    F.sum("ind_obito").alias("total_obitos"),
    F.round((F.sum("ind_obito") / F.count("*")) * 100, 2).alias("taxa_mortalidade_pct")
).orderBy(F.col("total_internacoes").desc()) \
 .withColumn("dh_carga_gold", F.current_timestamp())

df_gold_cid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.gold.gold_internacoes_por_diagnostico")

# Tabelas por Regiões

In [0]:
df_sih = spark.table("mvp_saude.silver.sih_cleaned")
df_municipios = spark.table("mvp_saude.silver.dim_municipios")

In [0]:
regioes_rs_completo = {
    "PORTO ALEGRE": "Metropolitana e Litoral", "CANOAS": "Metropolitana e Litoral",
    "GRAVATAI": "Metropolitana e Litoral", "GRAVATAÍ": "Metropolitana e Litoral",
    "NOVO HAMBURGO": "Metropolitana e Litoral", "SAO LEOPOLDO": "Metropolitana e Litoral",
    "ALVORADA": "Metropolitana e Litoral", "VIAMAO": "Metropolitana e Litoral",
    "VIAMÃO": "Metropolitana e Litoral", "CACHOEIRINHA": "Metropolitana e Litoral",
    "GUAIBA": "Metropolitana e Litoral", "ESTEIO": "Metropolitana e Litoral",
    "SAPUCAIA DO SUL": "Metropolitana e Litoral", "SAPIRANGA": "Metropolitana e Litoral",
    "CAMPO BOM": "Metropolitana e Litoral", "OSORIO": "Metropolitana e Litoral",
    "OSÓRIO": "Metropolitana e Litoral", "CAPAO DA CANOA": "Metropolitana e Litoral",
    "CAPÃO DA CANOA": "Metropolitana e Litoral", "TRAMANDAI": "Metropolitana e Litoral",
    "TRAMANDAÍ": "Metropolitana e Litoral", "TORRES": "Metropolitana e Litoral",

    "CAXIAS DO SUL": "Nordeste / Serra", "BENTO GONCALVES": "Nordeste / Serra",
    "BENTO GONÇALVES": "Nordeste / Serra", "GRAMADO": "Nordeste / Serra",
    "CANELA": "Nordeste / Serra", "FARROUPILHA": "Nordeste / Serra",
    "FLORES DA CUNHA": "Nordeste / Serra", "VACARIA": "Nordeste / Serra",
    "VERANOPOLIS": "Nordeste / Serra", "NOVA PRATA": "Nordeste / Serra",

    "LAJEADO": "Vales (Taquari e Rio Pardo)", "ESTRELA": "Vales (Taquari e Rio Pardo)",
    "TEUTONIA": "Vales (Taquari e Rio Pardo)", "TEUTÔNIA": "Vales (Taquari e Rio Pardo)",
    "ARROIO DO MEIO": "Vales (Taquari e Rio Pardo)", "SANTA CRUZ DO SUL": "Vales (Taquari e Rio Pardo)",
    "VENANCIO AIRES": "Vales (Taquari e Rio Pardo)", "VENÂNCIO AIRES": "Vales (Taquari e Rio Pardo)",
    "RIO PARDO": "Vales (Taquari e Rio Pardo)", "CACHOEIRA DO SUL": "Vales (Taquari e Rio Pardo)",
    "CRUZEIRO DO SUL": "Vales (Taquari e Rio Pardo)",

    "PELOTAS": "Sul", "RIO GRANDE": "Sul", "PIRATINI": "Sul",
    "SAO LOURENCO DO SUL": "Sul", "SÃO LOURENÇO DO SUL": "Sul",
    "CANGUCU": "Sul", "CANGUÇU": "Sul", "ARROIO GRANDE": "Sul",
    "SANTA VITORIA DO PALMAR": "Sul",

    "PASSO FUNDO": "Norte", "ERECHIM": "Norte", "CARAZINHO": "Norte",
    "MARAU": "Norte", "SANTO ANGELO": "Norte", "SANTO ÂNGELO": "Norte",
    "SOLEDADE": "Norte", "GETULIO VARGAS": "Norte",

    "IJUI": "Noroeste / Missões", "IJUÍ": "Noroeste / Missões",
    "SANTA ROSA": "Noroeste / Missões", "SAO BORJA": "Noroeste / Missões",
    "SÃO BORJA": "Noroeste / Missões", "TRES PASSOS": "Noroeste / Missões",
    "TRÊS PASSOS": "Noroeste / Missões", "HORIZONTINA": "Noroeste / Missões",

    "SANTA MARIA": "Central", "SANTIAGO": "Central",
    "CACAPAVA DO SUL": "Central", "CAÇAPAVA DO SUL": "Central",
    "JAGUARI": "Central", "RESTIGA SECA": "Central",

    "URUGUAIANA": "Fronteira Oeste", "ALEGRETE": "Fronteira Oeste",
    "SANTANA DO LIVRAMENTO": "Fronteira Oeste", "ITAQUI": "Fronteira Oeste",
    "QUARAI": "Fronteira Oeste", "QUARAÍ": "Fronteira Oeste",

    "BAGE": "Campanha", "BAGÉ": "Campanha", "DOM PEDRITO": "Campanha",
    "LAVRAS DO SUL": "Campanha"
}

In [0]:
map_data = [(k, v) for k, v in regioes_rs_completo.items()]
df_map_regioes = spark.createDataFrame(
    map_data, ["nome_municipio", "regiao_saude"]
)

In [0]:
df_gold_regioes = (
    df_sih.join(df_municipios, on="cod_municipio", how="inner")
    .withColumn(
        "nome_municipio_upper", F.upper(F.trim(F.col("nome_municipio")))
    )
    .join(
        df_map_regioes,
        F.col("nome_municipio_upper") == df_map_regioes["nome_municipio"],
        how="left",
    )
    .fillna({"regiao_saude": "Demais Municípios do Interior"})
    .groupBy("regiao_saude")
    .agg(
        F.sum("val_total_internacao").alias("custo_total_regiao"),
        F.count("*").alias("total_internacoes_regiao"),
        F.avg("val_total_internacao").alias("custo_medio_internacao_regiao"),
        F.sum("ind_obito").alias("total_obitos_regiao"),
    )
    .withColumn(
        "taxa_mortalidade_regiao_pct",
        (F.col("total_obitos_regiao") / F.col("total_internacoes_regiao")) * 100,
    )
    .orderBy(F.desc("custo_medio_internacao_regiao"))
)

In [0]:
df_gold_regioes.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("mvp_saude.gold.gold_custo_medio_por_regiao")